In [ ]:
# Import required modules
from fastdownload import FastDownload, download_url
from fastai.vision.all import *
from duckduckgo_search import DDGS
from urllib.error import HTTPError, URLError
from PIL import UnidentifiedImageError
from requests.exceptions import RequestException

# Use the duckduckgo search function
with DDGS() as ddgs:
    urls = ddgs.images("cat", max_results=5)
    for i, r in enumerate(urls):
        try:
            dest = f'cat{i}.jpg'  # Destination file name
            image_url = r['image']  # Image URL

            response = requests.get(image_url, allow_redirects=True)  # Get the image response
            if response.status_code == 403:
                continue
            
            download_url(r['image'], dest, show_progress = False) # Download the image using fastdownload
            print(r['image'], r['title']) # Print the image URL and title

            print(r['title']) # Print the title of the image
            print('-' * 40) # Print a separator line
            print()
            display(Image.open(dest).to_thumb(256, 256))   # Display the image using fastai's display function

        except Exception as e:
            print(f"An error occurred: {e}")
            continue

In [ ]:
search = 'cat' # Search term for images
path = Path('cat_or_not') # Path to save the images
dest = path/'cat' # Destination path for the downloaded images
dest.mkdir(parents=True, exist_ok=True) # Create the directory if it doesn't exist

with DDGS() as ddgs: 
    res = ddgs.images(search+' photo', max_results=200) # Search for images using duckduckgo
    urls = [r['image'] for r in res] # Get the image URLs

for i, url in enumerate(urls):
    try:
        download_url(url, dest/f"{i}.jpg", show_progress=False)
    except Exception as e:
        continue # Skip any exceptions that occur during download
    
time.sleep(1) # Sleep for 1 second to avoid overwhelming the server
resize_images(path, max_size=400, dest = path) # Resize the images to a maximum size of 400 pixels

failed = verify_images(get_image_files(path)) # Verify the images to check for any failed downloads
failed.map(Path.unlink) # Delete the failed images
print(len(failed), 'images failed to download.') # Print the number of images deleted

dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), # Define the data block with image and category blocks
    get_items=get_image_files, # Get the image files
    splitter=RandomSplitter(valid_pct=0.2, seed=42), # Split the data into training and validation sets
    get_y=parent_label, # Get the labels from the parent folder names
    item_tfms=[Resize(192, method='squish')] # Resizes and reshapes the images
).dataloaders(path/search, bs=32) # Create the data loaders with a batch size of 32

dls.show_batch(max_n=6) # Show a batch of images with their labels

In [ ]:
with DDGS() as ddgs:
    urls = ddgs.images("cat photo", max_results=1) # Search for images using duckduckgo
url = urls[0]['image'] # Get the first image URL
tempdest = 'cat.jpg' # Temporary destination for the image
download_url(url, tempdest, show_progress=False) # Download the image using fastdownload
display(Image.open(tempdest).to_thumb(256, 256)) # Display the downloaded image

learn = vision_learner(dls, resnet18, metrics=error_rate) # Create a vision learner using the data loaders and a pre-trained ResNet18 model
learn.fine_tune(3) # Fine-tune the model for 3 epoch

is_cat,_,probs = learn.predict(PILImage.create(tempdest)) # Make a prediction on the downloaded image
print(f"Prediction: {is_cat}, Probability: {probs[0]:.4f}") # Print the prediction and probability